In [ ]:
# === auto-inserted: bev-solving src on path ===
import sys, pathlib
_root = pathlib.Path.cwd()
while _root != _root.parent and not (_root / 'src' / 'geometry.py').exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


# train_v61_dinov2_lss_bevattn_kaggle

Kaggle notebook для продолжения `DINO + LSS` из старого `last.pt` с добавлением `BEV self-attention` в decoder bottleneck.


In [ ]:
from src.utils.dist import barrier, cleanup_distributed, get_local_rank, get_rank, get_world_size, is_dist_enabled, is_main_process, setup_distributed

%load_ext autoreload
%autoreload 2

import os, copy, time, json, math, random, gc
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from torchvision import transforms
from PIL import Image, ImageFile, ImageFilter
from tqdm import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

# Kaggle-safe dataset layout from letiti6e/bev-yandex
DATA_TRAIN = Path('/kaggle/input/bev-yandex/autonomy_yandex_dataset_train_kaggle_safe')
DATA_VAL   = Path('/kaggle/input/bev-yandex/autonomy_yandex_dataset_val_kaggle_safe')
DATA_TEST  = Path('/kaggle/input/bev-yandex/autonomy_yandex_dataset_test_kaggle_safe')

# Upload / mount your v6 last.pt in a Kaggle dataset and point here
ARTIFACTS_DIR = Path('/kaggle/input/bev-artifacts')
RESUME_CKPT = ARTIFACTS_DIR / 'last.pt'

cfg = {
    'run_dir': '/kaggle/working/runs/v62_dinov2_lss_bev2x_cleaned',
    'img_hw': (392, 700),
    'batch_size': 1,
    'val_batch_size': 1,
    'grad_accum': 24,
    'epochs': 40,
    'warmup_epochs': 0,
    'freeze_backbone_epochs': 0,
    'unfreeze_last_blocks': 2,
    'lr_backbone': 6e-6,
    'lr_image_neck': 6e-5,
    'lr_lss_bev': 8e-5,
    'weight_decay': 1e-4,
    'embed_weight_decay': 1e-2,
    'pos_weight': 5.0,
    'num_workers': 2,
    'val_num_workers': 0,
    'seed': 42,
    'ema_decay': 0.995,
    'val_target_size': 200,
    'min_rover_count': 30,
    'topk_rovers': 25,
    'rover_emb_dim': 8,
    'rover_cond_dim': 8,
    'mae_dedup_thr': 0.02,
    'dedup_camera': '/camera/inner/frontal/middle',
    'use_clean_cache': True,
    'backbone_name': 'dinov2_vitb14',
    'hub_repo': 'facebookresearch/dinov2',
    'backbone_out_dim': 768,
    'patch_size': 14,
    'tap_layers': [2, 5, 8, 11],
    'neck_dim': 128,
    'context_dim': 80,
    'depth_bins': 24,
    'depth_min': 1.0,
    'depth_max': 80.0,
    'world_z_min': -2.0,
    'world_z_max': 4.5,
    'bev_base_channels': 96,
    'bev_gn_groups': 8,
    'bev_upsample_factor': 2,
    'resume_training': True,
    'resume_ckpt': str(RESUME_CKPT),
    'use_ddp': False,
    'use_amp': True,
}

RUN_DIR = Path(cfg['run_dir'])
RUN_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = RUN_DIR / 'preproc_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
TORCH_HUB_DIR = RUN_DIR / 'torch_hub'
TORCH_HUB_DIR.mkdir(parents=True, exist_ok=True)
torch.hub.set_dir(str(TORCH_HUB_DIR))

setup_distributed()

random.seed(cfg['seed'] + get_rank())
np.random.seed(cfg['seed'] + get_rank())
torch.manual_seed(cfg['seed'] + get_rank())

if torch.cuda.is_available():
    device = torch.device(f'cuda:{get_local_rank()}') if is_dist_enabled() else torch.device('cuda')
elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

if is_main_process():
    print('device:', device)
    if torch.cuda.is_available():
        print('gpu:', torch.cuda.get_device_name(0))
    print('dataset exists:', DATA_TRAIN.exists(), DATA_VAL.exists(), DATA_TEST.exists())
    print('resume_ckpt exists:', RESUME_CKPT.exists(), RESUME_CKPT)

with open(RUN_DIR / 'config.json', 'w') as f:
    json.dump(json.loads(json.dumps(cfg, default=str)), f, indent=2)


In [ ]:
from src.geometry import kaggle_safe_name, load_info_with_root, remap_kaggle_paths, resolve_info_path, resolve_row_path

CAMERA_NAMES = [
    '/camera/inner/frontal/middle',
    '/camera/inner/frontal/far',
    '/side/left/forward',
    '/side/right/forward',
]
INTRINSICS_NAMES = [n + '/intrinsic_params' for n in CAMERA_NAMES]
CAR2CAM_NAMES = [n + '/car_to_cam' for n in CAMERA_NAMES]
GT_NAME = 'gt_occupancy_grid'

BEV_H, BEV_W = 188, 126
BEV_RES = 0.8
X_RANGE = (0.0, BEV_H * BEV_RES)
Y_RANGE = (-BEV_W * BEV_RES / 2, BEV_W * BEV_RES / 2)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.cleaning import build_img_hash, clean_merged_info, compute_gt_stats, smart_deduplicate


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.splits import build_rover_vocab_from_train, encode_rover, make_test_matched_split_target


In [ ]:
clean_info, dedup_report, clean_summary = clean_merged_info(
    DATA_TRAIN,
    DATA_VAL,
    cache_dir=CACHE_DIR,
    mae_thr=cfg['mae_dedup_thr'],
    dedup_camera=cfg['dedup_camera'],
    use_cache=cfg['use_clean_cache'],
)
clean_info = remap_kaggle_paths(clean_info, DATA_TRAIN, DATA_VAL, DATA_TEST)
print(json.dumps(clean_summary, indent=2, ensure_ascii=False))
display(clean_info.head(3))
if len(dedup_report):
    display(dedup_report.head(10))

train_idx, val_idx = make_test_matched_split_target(
    clean_info,
    DATA_TEST / 'info.csv',
    target_val_size=cfg['val_target_size'],
    seed=cfg['seed'],
    cache_path=CACHE_DIR / 'test_matched_val200_split.npz',
)
print('train_idx:', len(train_idx), 'val_idx:', len(val_idx))

train_info = clean_info.iloc[train_idx].reset_index(drop=True).copy()
val_info = clean_info.iloc[val_idx].reset_index(drop=True).copy()
train_info = remap_kaggle_paths(train_info, DATA_TRAIN, DATA_VAL, DATA_TEST)
val_info = remap_kaggle_paths(val_info, DATA_TRAIN, DATA_VAL, DATA_TEST)

rover_vocab, rover_stats = build_rover_vocab_from_train(
    train_info,
    min_count=cfg['min_rover_count'],
    topk=cfg['topk_rovers'],
)
rover_stats.to_csv(RUN_DIR / 'rover_embedding_stats.csv', index=False)
print('num rover classes including Other:', len(rover_vocab))
print('top rover mapping:', rover_vocab)
display(rover_stats.head(30))

if is_main_process() and len(val_info):
    row = val_info.iloc[0]
    for key in [CAMERA_NAMES[0], INTRINSICS_NAMES[0], CAR2CAM_NAMES[0], GT_NAME]:
        path = resolve_info_path(Path(row['__data_root']), row[key])
        print(key, path, path.exists())


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.data import BEVDatasetV4Clean


In [ ]:
from src.models.lss import ASPP2d, ConvGNAct, LSSViewTransform2D, ResidualBlock2d, _DINOv2MultiScaleBackbone, gn_groups

def load_warm_start(model: nn.Module, ckpt_path: str | Path):
    ckpt_path = Path(ckpt_path)
    if not ckpt_path.exists():
        print('warm-start checkpoint not found, starting from random init:', ckpt_path)
        return {'loaded': 0, 'skipped': 0, 'missing': None, 'unexpected': None}

    ckpt = torch.load(ckpt_path, map_location='cpu')
    state = ckpt['ema'] if 'ema' in ckpt else ckpt.get('model', ckpt)
    cur = model.state_dict()
    loadable = {}
    skipped = []
    for k, v in state.items():
        if k in cur and cur[k].shape == v.shape:
            loadable[k] = v
        else:
            skipped.append(k)
    missing, unexpected = model.load_state_dict(loadable, strict=False)
    print('loaded warm-start from', ckpt_path)
    print('  matched keys:', len(loadable))
    print('  skipped old keys:', len(skipped))
    print('  missing in new model:', len(missing))
    print('  unexpected during load:', len(unexpected))
    if len(skipped):
        print('  sample skipped:', skipped[:10])
    return {
        'loaded': len(loadable),
        'skipped': len(skipped),
        'missing': missing,
        'unexpected': unexpected,
    }

class StrongBEVEncoderDecoderHiRes(nn.Module):
    def __init__(self, in_c: int, base_c: int = 96, groups: int = 8, out_down_factor: int = 2):
        super().__init__()
        self.out_down_factor = int(out_down_factor)
        self.stem = nn.Sequential(
            ConvGNAct(in_c, base_c, k=3, s=1, p=1, groups=groups, act=True),
            ResidualBlock2d(base_c, base_c, stride=1, groups=groups),
        )
        self.down1 = nn.Sequential(
            ResidualBlock2d(base_c, base_c * 2, stride=2, groups=groups),
            ResidualBlock2d(base_c * 2, base_c * 2, stride=1, groups=groups),
        )
        self.down2 = nn.Sequential(
            ResidualBlock2d(base_c * 2, base_c * 4, stride=2, groups=groups),
            ResidualBlock2d(base_c * 4, base_c * 4, stride=1, groups=groups),
        )
        self.aspp = ASPP2d(base_c * 4, base_c * 4, rates=(1, 3, 6), groups=groups)
        self.up1 = nn.Sequential(
            ConvGNAct(base_c * 4 + base_c * 2, base_c * 2, k=3, s=1, p=1, groups=groups, act=True),
            ResidualBlock2d(base_c * 2, base_c * 2, stride=1, groups=groups),
        )
        self.up0 = nn.Sequential(
            ConvGNAct(base_c * 2 + base_c, base_c, k=3, s=1, p=1, groups=groups, act=True),
            ResidualBlock2d(base_c, base_c, stride=1, groups=groups),
        )
        self.head = nn.Conv2d(base_c, 1, 1)

    def forward_debug(self, x):
        s0 = self.stem(x)
        s1 = self.down1(s0)
        s2 = self.down2(s1)
        b = self.aspp(s2)
        u1 = self.up1(torch.cat([F.interpolate(b, size=s1.shape[-2:], mode='bilinear', align_corners=False), s1], dim=1))
        u0 = self.up0(torch.cat([F.interpolate(u1, size=s0.shape[-2:], mode='bilinear', align_corners=False), s0], dim=1))
        logits_hr = self.head(u0)
        if self.out_down_factor > 1:
            logits = F.avg_pool2d(logits_hr, kernel_size=self.out_down_factor, stride=self.out_down_factor)
        else:
            logits = logits_hr
        return {'logits_hr': logits_hr, 'logits': logits}

    def forward(self, x):
        return self.forward_debug(x)['logits']

class MultiCamBEVv62DINOv2LSSBEV2xClean(nn.Module):
    def __init__(self, num_rover_classes: int,
                 rover_emb_dim: int = 8,
                 rover_cond_dim: int = 8,
                 n_cameras: int = 4,
                 freeze_backbone: bool = False,
                 hub_repo: str = 'facebookresearch/dinov2',
                 backbone_name: str = 'dinov2_vitb14',
                 backbone_out_dim: int = 768,
                 patch_size: int = 14,
                 tap_layers=(2, 5, 8, 11),
                 neck_dim: int = 128,
                 context_dim: int = 80,
                 depth_bins: int = 24,
                 depth_min: float = 1.0,
                 depth_max: float = 80.0,
                 world_z_min: float = -2.0,
                 world_z_max: float = 4.5,
                 bev_base_channels: int = 96,
                 bev_gn_groups: int = 8,
                 bev_upsample_factor: int = 2):
        super().__init__()
        self.n_cameras = n_cameras
        self.rover_cond_dim = rover_cond_dim
        self.bev_upsample_factor = int(bev_upsample_factor)
        self.bev_h_hr = BEV_H * self.bev_upsample_factor
        self.bev_w_hr = BEV_W * self.bev_upsample_factor
        self.bev_res_hr = BEV_RES / float(self.bev_upsample_factor)

        self.backbone = _DINOv2MultiScaleBackbone(
            hub_repo=hub_repo,
            backbone_name=backbone_name,
            out_dim=backbone_out_dim,
            patch_size=patch_size,
            tap_layers=tap_layers,
            neck_dim=neck_dim,
            groups=bev_gn_groups,
        )
        if freeze_backbone:
            for p in self.backbone.vit.parameters():
                p.requires_grad = False

        self.view_transform = LSSViewTransform2D(
            in_c=neck_dim,
            context_c=context_dim,
            depth_bins=depth_bins,
            depth_min=depth_min,
            depth_max=depth_max,
            bev_h=self.bev_h_hr,
            bev_w=self.bev_w_hr,
            bev_res=self.bev_res_hr,
            x_range=X_RANGE,
            y_range=Y_RANGE,
            z_min=world_z_min,
            z_max=world_z_max,
            groups=bev_gn_groups,
        )

        self.rover_embed = nn.Embedding(num_rover_classes, rover_emb_dim)
        nn.init.normal_(self.rover_embed.weight, std=0.02)
        self.rover_mlp = nn.Sequential(
            nn.Linear(rover_emb_dim, 16),
            nn.ReLU(inplace=True),
            nn.Linear(16, rover_cond_dim),
            nn.ReLU(inplace=True),
        )

        self.bev_decoder = StrongBEVEncoderDecoderHiRes(
            in_c=context_dim + rover_cond_dim,
            base_c=bev_base_channels,
            groups=bev_gn_groups,
            out_down_factor=self.bev_upsample_factor,
        )

    def forward_debug(self, images, intrinsics, car2cams, rover_ids):
        B, N, C, H, W = images.shape
        assert N == self.n_cameras
        x = images.reshape(B * N, C, H, W)
        back = self.backbone(x)
        fused = back['fused'].reshape(B, N, back['fused'].shape[1], back['fused'].shape[2], back['fused'].shape[3])
        bev_hr, vt_debug = self.view_transform(fused, intrinsics, car2cams, image_hw=(H, W))
        rover_feat = self.rover_mlp(self.rover_embed(rover_ids)).view(B, self.rover_cond_dim, 1, 1)
        rover_map = rover_feat.expand(-1, -1, self.bev_h_hr, self.bev_w_hr)
        dec_debug = self.bev_decoder.forward_debug(torch.cat([bev_hr, rover_map], dim=1))
        return {
            'tap_features': back['tap_features'],
            'image_fused': fused,
            'depth_logits': vt_debug['depth_logits'],
            'depth_prob': vt_debug['depth_prob'],
            'bev_raw_hr': vt_debug['bev'],
            'valid_ratio': vt_debug['valid_ratio'],
            'logits_hr': dec_debug['logits_hr'],
            'logits': dec_debug['logits'],
        }

    def forward(self, images, intrinsics, car2cams, rover_ids):
        dbg = self.forward_debug(images, intrinsics, car2cams, rover_ids)
        logits = torch.nan_to_num(dbg['logits'], nan=0.0, posinf=0.0, neginf=0.0)
        return logits

def load_hr_resume_state(core_model, ema_model, run_dir: Path):
    resume_ckpt = Path(cfg.get('resume_ckpt', ''))
    if not cfg.get('resume_training', False) or not resume_ckpt.exists():
        return {
            'enabled': False,
            'start_epoch': 0,
            'best_iou': -1.0,
            'best_ema_iou': -1.0,
            'log': [],
            'elapsed_minutes': 0.0,
        }

    ckpt = torch.load(resume_ckpt, map_location='cpu')
    core_model.load_state_dict(ckpt['model'], strict=True)
    if 'ema' in ckpt:
        ema_model.load_state_dict(ckpt['ema'], strict=True)
    else:
        ema_model.load_state_dict(ckpt['model'], strict=True)

    start_epoch = int(ckpt.get('epoch', -1)) + 1
    log_path = resume_ckpt.parent / 'log.csv'
    log_rows = []
    elapsed_minutes = 0.0
    if log_path.exists():
        log_rows = pd.read_csv(log_path).to_dict('records')
        if len(log_rows):
            elapsed_minutes = float(log_rows[-1].get('minutes', 0.0) or 0.0)

    best_iou = float(ckpt.get('best_iou', -1.0))
    best_ema_iou = float(ckpt.get('best_ema_iou', -1.0))
    print('high-res resumed from', resume_ckpt)
    print('  strict model/ema load succeeded')
    print('  start_epoch:', start_epoch)

    return {
        'enabled': True,
        'start_epoch': start_epoch,
        'best_iou': best_iou,
        'best_ema_iou': best_ema_iou,
        'log': log_rows,
        'elapsed_minutes': elapsed_minutes,
    }


In [ ]:
from src.losses import CompoundLossV2, _lovasz_grad, lovasz_hinge_flat
from src.metrics import iou_binary_batch
from src.utils.training import cleanup_cuda, unwrap_model

@torch.inference_mode()
def evaluate_iou(model, loader, threshold=0.5, desc='val@0.5'):
    model.eval()
    inter = 0
    union = 0
    pbar = tqdm(loader, desc=desc, leave=False)
    for batch in pbar:
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt = batch['gt'].to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda' and cfg['use_amp'])):
            logits = model(images, intr, c2c, rover_id)
        logits = logits.float()
        i, u = iou_binary_batch(logits, gt, threshold=threshold)
        inter += i
        union += u
        pbar.set_postfix(iou=f'{inter / max(union, 1):.4f}')
    return inter / max(union, 1)


In [ ]:
from src.utils.training import freeze_all_backbone, unfreeze_last_blocks, update_ema

ds_train_warm = BEVDatasetV4Clean(train_info, mode='train', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab)
ds_train_aug = BEVDatasetV4Clean(train_info, mode='train', img_hw=cfg['img_hw'], aug=True, rover_vocab=rover_vocab)
ds_val = BEVDatasetV4Clean(val_info, mode='val', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab)

train_sampler_warm = None
train_sampler_aug = None
if is_dist_enabled():
    train_sampler_warm = DistributedSampler(ds_train_warm, num_replicas=get_world_size(), rank=get_rank(), shuffle=True, drop_last=True)
    train_sampler_aug = DistributedSampler(ds_train_aug, num_replicas=get_world_size(), rank=get_rank(), shuffle=True, drop_last=True)

loader_warm = DataLoader(ds_train_warm, batch_size=cfg['batch_size'], shuffle=(train_sampler_warm is None), sampler=train_sampler_warm, num_workers=cfg['num_workers'], pin_memory=(device.type == 'cuda'), drop_last=True)
loader_train = DataLoader(ds_train_aug, batch_size=cfg['batch_size'], shuffle=(train_sampler_aug is None), sampler=train_sampler_aug, num_workers=cfg['num_workers'], pin_memory=(device.type == 'cuda'), drop_last=True)
loader_val = None
if is_main_process():
    loader_val = DataLoader(ds_val, batch_size=cfg['val_batch_size'], shuffle=False, num_workers=cfg['val_num_workers'], pin_memory=(device.type == 'cuda'))
    print('loader_train batch_size:', loader_train.batch_size, '| workers:', loader_train.num_workers)

base_model = MultiCamBEVv62DINOv2LSSBEV2xClean(
    num_rover_classes=len(rover_vocab),
    rover_emb_dim=cfg['rover_emb_dim'],
    rover_cond_dim=cfg['rover_cond_dim'],
    freeze_backbone=False,
    hub_repo=cfg['hub_repo'],
    backbone_name=cfg['backbone_name'],
    backbone_out_dim=cfg['backbone_out_dim'],
    patch_size=cfg['patch_size'],
    tap_layers=cfg['tap_layers'],
    neck_dim=cfg['neck_dim'],
    context_dim=cfg['context_dim'],
    depth_bins=cfg['depth_bins'],
    depth_min=cfg['depth_min'],
    depth_max=cfg['depth_max'],
    world_z_min=cfg['world_z_min'],
    world_z_max=cfg['world_z_max'],
    bev_base_channels=cfg['bev_base_channels'],
    bev_gn_groups=cfg['bev_gn_groups'],
    bev_upsample_factor=cfg['bev_upsample_factor'],
).to(device)

freeze_all_backbone(base_model)

if is_dist_enabled():
    model = DDP(base_model, device_ids=[get_local_rank()], output_device=get_local_rank(), broadcast_buffers=False, find_unused_parameters=False)
else:
    model = base_model

core_model = unwrap_model(model)
backbone_params = []
embed_params = []
image_neck_params = []
other_params = []
for name, p in core_model.named_parameters():
    if name.startswith('backbone.vit.'):
        backbone_params.append(p)
    elif name.startswith('rover_embed.'):
        embed_params.append(p)
    elif name.startswith('backbone.laterals.') or name.startswith('backbone.fuse.') or name.startswith('backbone.down') or name.startswith('backbone.neck_out.'):
        image_neck_params.append(p)
    else:
        other_params.append(p)

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': cfg['lr_backbone'], 'weight_decay': cfg['weight_decay']},
    {'params': image_neck_params, 'lr': cfg['lr_image_neck'], 'weight_decay': cfg['weight_decay']},
    {'params': other_params, 'lr': cfg['lr_lss_bev'], 'weight_decay': cfg['weight_decay']},
    {'params': embed_params, 'lr': cfg['lr_lss_bev'], 'weight_decay': cfg['embed_weight_decay']},
])
remaining_epochs = max(cfg['epochs'] - max(0, int(torch.load(cfg['resume_ckpt'], map_location='cpu').get('epoch', -1)) + 1), 1) if Path(cfg['resume_ckpt']).exists() else cfg['epochs']
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=remaining_epochs, eta_min=1e-6)
loss_fn = CompoundLossV2(pos_weight=cfg['pos_weight']).to(device)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda' and cfg['use_amp']))

ema_model = copy.deepcopy(core_model).to(device).eval()
for p in ema_model.parameters():
    p.requires_grad = False

resume_state = load_hr_resume_state(core_model, ema_model, RUN_DIR)
unfreeze_last_blocks(model, n_last_blocks=cfg['unfreeze_last_blocks'])

if is_main_process():
    print('params M total:', sum(p.numel() for p in core_model.parameters()) / 1e6)
    print('bev upsample factor:', cfg['bev_upsample_factor'])
    print('resume enabled:', resume_state['enabled'])
    print('resume start_epoch:', resume_state['start_epoch'])

    with torch.no_grad():
        batch = next(iter(loader_train))
        images = batch['images'].to(device)
        intr = batch['intrinsics'].to(device)
        c2c = batch['car2cams'].to(device)
        rover_id = batch['rover_id'].to(device)
        dbg = core_model.forward_debug(images, intr, c2c, rover_id)
        print('image_fused:', tuple(dbg['image_fused'].shape), torch.isfinite(dbg['image_fused']).all().item())
        print('bev_raw_hr:', tuple(dbg['bev_raw_hr'].shape), torch.isfinite(dbg['bev_raw_hr']).all().item())
        print('logits_hr:', tuple(dbg['logits_hr'].shape), torch.isfinite(dbg['logits_hr']).all().item())
        print('logits:', tuple(dbg['logits'].shape), torch.isfinite(dbg['logits']).all().item())

cleanup_cuda()
barrier()


In [ ]:
log = list(resume_state['log'])
best_iou = float(resume_state['best_iou'])
best_ema_iou = float(resume_state['best_ema_iou'])
start = time.time() - float(resume_state['elapsed_minutes']) * 60.0
start_epoch = int(resume_state['start_epoch'])

for epoch in range(start_epoch, cfg['epochs']):
    if train_sampler_warm is not None:
        train_sampler_warm.set_epoch(epoch)
    if train_sampler_aug is not None:
        train_sampler_aug.set_epoch(epoch)

    model.train()
    loader = loader_train
    losses = []
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(loader, desc=f'ep{epoch:02d} [BEV2X]', disable=not is_main_process())

    for step, batch in enumerate(pbar):
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt = batch['gt'].to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda' and cfg['use_amp'])):
            logits = model(images, intr, c2c, rover_id)
        logits = logits.float()
        loss, parts = loss_fn(logits, gt)
        loss = loss / cfg['grad_accum']

        if not torch.isfinite(loss):
            raise RuntimeError(f'Non-finite loss detected at epoch={epoch} step={step}: {loss.item()}')

        scaler.scale(loss).backward()
        if (step + 1) % cfg['grad_accum'] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(core_model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            update_ema(ema_model, model, cfg['ema_decay'])

        losses.append(loss.item() * cfg['grad_accum'])
        if is_main_process() and step % 20 == 0:
            pbar.set_postfix(loss=f'{np.mean(losses[-50:]):.3f}', bce=f"{parts['bce']:.2f}")

    if len(loader) % cfg['grad_accum'] != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(core_model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        update_ema(ema_model, model, cfg['ema_decay'])

    scheduler.step()
    barrier()

    if is_main_process():
        cleanup_cuda()
        print('start val model @0.5')
        val_iou_05 = evaluate_iou(core_model, loader_val, threshold=0.5, desc=f'val model ep{epoch:02d}')

        cleanup_cuda()
        print('start val ema @0.5')
        val_iou_05_ema = evaluate_iou(ema_model, loader_val, threshold=0.5, desc=f'val ema ep{epoch:02d}')
        cleanup_cuda()

        elapsed = (time.time() - start) / 60
        row = {
            'epoch': epoch,
            'loss': float(np.mean(losses)) if len(losses) else None,
            'val_iou_05': float(val_iou_05),
            'val_iou_05_ema': float(val_iou_05_ema),
            'minutes': float(elapsed),
        }

        print(f"ep{epoch:02d} | loss {np.mean(losses):.3f} | val@0.5 {val_iou_05:.4f} | ema@0.5 {val_iou_05_ema:.4f} | {elapsed:.1f}m")

        if val_iou_05 > best_iou:
            best_iou = val_iou_05
            torch.save({
                'model_type': 'v62_dinov2_lss_bev2x_cleaned',
                'model': core_model.state_dict(),
                'epoch': epoch,
                'best_iou': float(val_iou_05),
                'best_t': 0.5,
                'rover_vocab': rover_vocab,
                'cfg': cfg,
            }, RUN_DIR / 'best.pt')
            print('  new best model:', val_iou_05)

        if val_iou_05_ema > best_ema_iou:
            best_ema_iou = val_iou_05_ema
            torch.save({
                'model_type': 'v62_dinov2_lss_bev2x_cleaned',
                'model': core_model.state_dict(),
                'ema': ema_model.state_dict(),
                'epoch': epoch,
                'best_ema_iou': float(val_iou_05_ema),
                'best_t_ema': 0.5,
                'rover_vocab': rover_vocab,
                'cfg': cfg,
            }, RUN_DIR / 'ema_best.pt')
            print('  new best ema:', val_iou_05_ema)

        log.append(row)
        pd.DataFrame(log).to_csv(RUN_DIR / 'log.csv', index=False)
        torch.save({
            'model_type': 'v62_dinov2_lss_bev2x_cleaned',
            'model': core_model.state_dict(),
            'ema': ema_model.state_dict(),
            'epoch': epoch,
            'best_iou': float(best_iou),
            'best_ema_iou': float(best_ema_iou),
            'rover_vocab': rover_vocab,
            'cfg': cfg,
        }, RUN_DIR / 'last.pt')

    barrier()


### Notes

- Это **architectural upgrade resume**, а не чистый resume: веса модели и EMA подгружаются из старого `last.pt` только по совпадающим ключам.
- Новые `BEV attention` блоки инициализируются случайно и учатся с отдельным `lr_new_attn`.
- Optimizer/scheduler/scaler специально стартуют заново: их shape/state из старой архитектуры не переносится.
- По умолчанию notebook заточен под kaggle-safe dataset layout и один GPU.
